In [10]:

from datetime import datetime, timezone
import time
from oandapyV20 import API
import oandapyV20.endpoints.positions as posit
import oandapyV20.endpoints.orders as orders
import oandapyV20.endpoints.trades as trades
import logging

import config
import oanda_api_handler as oanda
import EMA_Trend_following as strategy
from pyTrade_validation import Validation

In [11]:
retries = 2
data = {
        "longUnits": "ALL",
        "shortUnits": "ALL"
    }

client =  API(access_token=config.OANDA_API_KEY, environment=config.OANDA_ENVIRON)
accountID = config.OANDA_ACCOUNT_ID

In [12]:
positions = oanda.open_positions(client)
print('Checking for positions....')
if not positions:
    print("No open positions.")
else:
    print(f"Open positions: {len(positions)}")

    for position in positions:
        instrument = position.get("instrument", "Unknown")
        print(f"\nInstrument: {instrument}")

        for side in ("long", "short"):
            details = position.get(side, {})
            units = details.get("units", "0")

            if units != "0":
                print(
                    f"  {side.capitalize()}: "
                    f"units={units}, "
                    f"averagePrice={details.get('averagePrice', 'N/A')}, "
                    f"unrealizedPL={details.get('unrealizedPL', 'N/A')}"
                )

        trade_ids = details.get("tradeIDs", [])
        print(
            f"  {side.capitalize()} trade IDs: "
            f"{', '.join(map(str, trade_ids)) if trade_ids else 'N/A'}"
        )



Checking for positions....
Open positions: 2

Instrument: AUD_USD
  Long: units=99, averagePrice=0.70262, unrealizedPL=-0.0310
  Short trade IDs: N/A

Instrument: EUR_USD
  Long: units=99, averagePrice=1.14089, unrealizedPL=-0.1381
  Short trade IDs: N/A


In [13]:
r = orders.OrdersPending(accountID)
order_list = client.request(r)['orders']
if not order_list:
    print("No pending orders.")
else:
    print(f"Pending orders: {len(order_list)}")
    for pending_order in order_list:
        print(
            f"ID: {pending_order.get('id')} | "
            f"Type: {pending_order.get('type')} | "
            f"Trade: {pending_order.get('tradeID')} | "
            f"Price: {pending_order.get('price')} | "
            f"State: {pending_order.get('state')}"
        )

Pending orders: 2
ID: 818 | Type: STOP_LOSS | Trade: 816 | Price: 1.09100 | State: PENDING
ID: 817 | Type: TAKE_PROFIT | Trade: 816 | Price: 1.24100 | State: PENDING


In [14]:
r = trades.OpenTrades(config.OANDA_ACCOUNT_ID)
trade_list = client.request(r)["trades"]
if not trade_list:
    print("No open trades found.")
else:
    print(f"Open trades: {len(trade_list)}")
    for trade in trade_list:
        print(
            f"ID: {trade.get('id', 'N/A')} | "
            f"Instrument: {trade.get('instrument', 'N/A')} | "
            f"Units: {trade.get('currentUnits', 'N/A')} | "
            f"Price: {trade.get('price', 'N/A')} | "
            f"State: {trade.get('state', 'N/A')}"
        )

Open trades: 2
ID: 816 | Instrument: EUR_USD | Units: 99 | Price: 1.14089 | State: OPEN
ID: 796 | Instrument: AUD_USD | Units: 99 | Price: 0.70262 | State: OPEN


In [15]:
#check for tradeIDs by position in the list pending orders
